In [1]:
import random
import re
import os
from pathlib import Path
from moviepy.video.VideoClip import ImageClip
from moviepy import VideoFileClip, AudioFileClip, concatenate_videoclips, CompositeVideoClip
from pocket_tts import TTSModel
import scipy.io.wavfile
from scipy.io import wavfile
import wave
import contextlib
import soundfile as sf
import numpy as np
from datetime import timedelta
from moviepy.video.tools.subtitles import SubtitlesClip
from moviepy.video.VideoClip import TextClip
import unicodedata
from moviepy.audio.io.AudioFileClip import AudioFileClip

c:\Users\lidia\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
file_path = r'D:\1Graduation project\ECHO\experiments\video_generation\video_generation_pharaohs\Scripts\Ramsess II.txt'
with open(file_path, 'r', encoding='utf-8') as file:
    text = file.read()

paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
paragraphs

['Rameses II was one of ancient Egypt’s greatest pharaohs. Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war. At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side. He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.',
 'Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region. This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns. Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.',
 "He continued his father’s restoration projects throughout Egypt and Nubia, reopening mines and quarries to ensure the empire's strength. Rameses faced a legendary battle against the Hittites that ended in stalemate but led to history’s first known peace treaty between two great powers."

In [10]:
# Folder with images
image_folder = Path(r"D:\1Graduation project\ECHO\data\video_generation\raw\pharaohs_images\Ramesses II")
image_files = list(image_folder.glob("*.jpg")) + list(image_folder.glob("*.jpeg")) + list(image_folder.glob("*.png")) + list(image_folder.glob("*.heic"))

# Output folder
output_folder = Path(r"D:\1Graduation project\ECHO\data\video_generation\outputs\Output_Videos")
output_folder.mkdir(exist_ok=True)

In [ ]:
#Text-to-Speech
tts_model = TTSModel.load_model()
voice_state = tts_model.get_state_for_audio_prompt("alba")

i = 0
for p in paragraphs:
    audio = tts_model.generate_audio(voice_state, p)
    scipy.io.wavfile.write(f"output_{i}.wav", tts_model.sample_rate, audio.numpy())
    i += 1

c:\Users\lidia\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lidia\.cache\huggingface\hub\models--kyutai--pocket-tts-without-voice-cloning. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [5]:
wav_files = sorted([f for f in os.listdir('.') if f.endswith('.wav')])
print(wav_files)

audio_data = []
samplerate = None

for file_name in wav_files:
    data, sr = sf.read(file_name)
    
    if samplerate is None:
        samplerate = sr
    elif sr != samplerate:
        raise ValueError("Sample rates do not match!")

    audio_data.append(data)

# Concatenate along time axis
combined = np.concatenate(audio_data, axis=0)

# Write output (keeps float format safely)
sf.write("Rameses II.wav", combined, samplerate)

['output_0.wav', 'output_1.wav', 'output_2.wav', 'output_3.wav', 'output_4.wav']


In [6]:
durations = []
images_needed = []
seconds = []
for i in range(len(paragraphs)):
    file_path = f"output_{i}.wav"
    # Returns the sample rate (Fs) and the data as a NumPy array
    Fs, data = wavfile.read(file_path) 
    # The length of the data array divided by the sample rate gives the duration
    duration_seconds = len(data) / float(Fs)
    durations.append(duration_seconds)
    images_needed.append(max(1, int(duration_seconds / 6)))
    section_seconds = duration_seconds / images_needed[-1]
    for img in range(images_needed[-1]):
        seconds.append(section_seconds)

print(f"Durations of audio files: {durations}")
print(f"Images needed for each paragraph: {images_needed}")
print(f"Seconds for each image: {seconds}")

Durations of audio files: [21.68, 19.44, 20.72, 20.72, 19.6]
Images needed for each paragraph: [3, 3, 3, 3, 3]
Seconds for each image: [7.226666666666667, 7.226666666666667, 7.226666666666667, 6.48, 6.48, 6.48, 6.906666666666666, 6.906666666666666, 6.906666666666666, 6.906666666666666, 6.906666666666666, 6.906666666666666, 6.533333333333334, 6.533333333333334, 6.533333333333334]


In [7]:
# ---------- SETTINGS ----------
MAX_CHARS_PER_LINE = 42
MAX_LINES = 2
MIN_DURATION = 1.0  # minimum subtitle duration in seconds


# ---------- HELPERS ----------
def normalize_text(text):
    """
    Cleans problematic Unicode characters that break
    subtitle rendering in Pillow / MoviePy.
    """

    # First normalize Unicode composition
    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "’": "'",
        "‘": "'",
        "‚": ",",
        "‛": "'",

        "“": '"',
        "”": '"',
        "„": '"',

        "—": "-",
        "–": "-",
        "―": "-",

        "…": "...",

        "\u00A0": " ",   # non-breaking space
        "\u200B": "",    # zero-width space
        "\u200C": "",
        "\u200D": "",
        "\uFEFF": "",    # BOM if embedded
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    # Remove any remaining control characters
    text = "".join(
        ch for ch in text
        if unicodedata.category(ch)[0] != "C"
    )

    return text

def format_timestamp(seconds):
    td = timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    millis = int((seconds - total_seconds) * 1000)

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    secs = total_seconds % 60

    return f"{hours:02}:{minutes:02}:{secs:02},{millis:03}"


def split_into_sentences(text):
    return re.split(r'(?<=[.!?]) +', text.strip())


def split_long_text(text, max_chars=MAX_CHARS_PER_LINE):
    words = text.split()
    lines = []
    current_line = ""

    for word in words:
        if len(current_line) + len(word) + 1 <= max_chars:
            current_line += (" " if current_line else "") + word
        else:
            lines.append(current_line)
            current_line = word

    if current_line:
        lines.append(current_line)

    # combine into blocks of max 2 lines
    blocks = []
    for i in range(0, len(lines), MAX_LINES):
        block = "\n".join(lines[i:i + MAX_LINES])
        blocks.append(block)

    return blocks


# ---------- MAIN FUNCTION ----------
def generate_srt(paragraphs, durations, output_path="subtitles.srt"):
    assert len(paragraphs) == len(durations), "Paragraphs and durations must match"

    current_time = 0.0
    subtitle_index = 1
    srt_blocks = []

    for paragraph, duration in zip(paragraphs, durations):
        paragraph = normalize_text(paragraph)
        sentences = split_into_sentences(paragraph)

        # break into subtitle chunks
        chunks = []
        for sentence in sentences:
            chunks.extend(split_long_text(sentence))

        total_chars = sum(len(chunk.replace("\n", "")) for chunk in chunks)

        for chunk in chunks:
            chunk_char_count = len(chunk.replace("\n", ""))

            # proportional timing
            chunk_duration = max(
                MIN_DURATION,
                (chunk_char_count / total_chars) * duration
            )

            start_time = current_time
            end_time = current_time + chunk_duration

            srt_block = f"""{subtitle_index}
{format_timestamp(start_time)} --> {format_timestamp(end_time)}
{chunk}

"""
            srt_blocks.append(srt_block)

            current_time = end_time
            subtitle_index += 1

    with open(output_path, "w", encoding="utf-8-sig") as f:
        f.writelines(srt_blocks)

    print(f"SRT file saved to {output_path}")


generate_srt(paragraphs, durations, "output_subtitles.srt")

SRT file saved to output_subtitles.srt


In [8]:
fps = 30
target_size = (1920, 1080)

def ken_burns(clip, duration, zoom_factor=1.15):
    target_w, target_h = target_size

    # ---- 1. Scale to fully cover frame with zoom headroom ----
    scale = max(target_w / clip.w, target_h / clip.h) * zoom_factor
    clip = clip.resized(scale)

    max_x = max(0, clip.w - target_w)
    max_y = max(0, clip.h - target_h)

    # ---- 2. Choose movement direction based on aspect ratio ----
    aspect_ratio = clip.w / clip.h
    target_ratio = target_w / target_h

    if aspect_ratio > target_ratio:
        # Image is wider → move horizontally
        direction = "horizontal"
    else:
        # Image is taller → move vertically
        direction = "vertical"

    # ---- 3. Define safe start/end positions ----
    if direction == "horizontal" and max_x > 0:
        start_x = 0
        end_x = -max_x
        start_y = -max_y / 2
        end_y = start_y

    elif direction == "vertical" and max_y > 0:
        start_y = 0
        end_y = -max_y
        start_x = -max_x / 2
        end_x = start_x

    else:
        # No room to move safely — just center it
        start_x = end_x = -max_x / 2
        start_y = end_y = -max_y / 2

    # ---- 4. Position function with safety clamp ----
    def position(t):
        progress = min(max(t / duration, 0), 1)

        x = start_x + (end_x - start_x) * progress
        y = start_y + (end_y - start_y) * progress

        # Clamp to ensure no black edges ever appear
        x = min(0, max(-max_x, x))
        y = min(0, max(-max_y, y))

        return (x, y)

    moving = clip.with_position(position).with_duration(duration)

    final = CompositeVideoClip(
        [moving],
        size=target_size
    ).with_duration(duration)

    return final

clips = []
i = 0
for img_path in image_files:
    print(img_path)
    
    clip_duration = seconds[i]
    i += 1

    clip = ImageClip(str(img_path)).with_duration(clip_duration)
    
    clip = ken_burns(clip, clip_duration)
    clips.append(clip)

C:\Uni\GP\Code\Generation\Rameses II images\1.Rameses II passport.jpg
C:\Uni\GP\Code\Generation\Rameses II images\11.Ramesseum.jpg
C:\Uni\GP\Code\Generation\Rameses II images\15.The sherden.jpg
C:\Uni\GP\Code\Generation\Rameses II images\4.Battle of Kadesh.jpg
C:\Uni\GP\Code\Generation\Rameses II images\5.Rameses II with a prisoner painted relief.jpg
C:\Uni\GP\Code\Generation\Rameses II images\6.Peace treaty of the Hitties and Rameses II.jpg
C:\Uni\GP\Code\Generation\Rameses II images\8.Pi-Ramesses.jpg
C:\Uni\GP\Code\Generation\Rameses II images\10.Rameses II tomb at valley of the kings.png
C:\Uni\GP\Code\Generation\Rameses II images\12.Rameses II mummy.png
C:\Uni\GP\Code\Generation\Rameses II images\14.Rameses II relief.png
C:\Uni\GP\Code\Generation\Rameses II images\7.Abu Simbel.png
C:\Uni\GP\Code\Generation\Rameses II images\9.Rameses II statue from temple.png
C:\Uni\GP\Code\Generation\Rameses II images\13.Statue 4.HEIC
C:\Uni\GP\Code\Generation\Rameses II images\2.Statue 2.HEIC
C:\

In [10]:
final_video = concatenate_videoclips(clips, method="compose")

In [11]:
audio = AudioFileClip("Rameses II.wav")
final_video = final_video.with_audio(audio)

In [ ]:
final_video.write_videofile(
    "video_no_subs.mp4",
    fps=fps,
    codec="libx264",
    audio_codec="aac",
    preset="ultrafast",
    threads=8
)

MoviePy - Building video video_no_subs.mp4.
MoviePy - Writing audio in video_no_subsTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video video_no_subs.mp4



MoviePy - Done !
MoviePy - video ready video_no_subs.mp4


In [ ]:
video = final_video  # reuse object

def subtitle_generator(txt):
    return TextClip(
        text=txt,
        font="C:/Windows/Fonts/arial.ttf",
        font_size=40,
        color="white",
        stroke_color="black",
        stroke_width=2,
        method="caption",
        size=(int(video.w * 0.8), None),
        text_align="center",
        horizontal_align="center",
        margin=(20, 20)
    )

subtitles = SubtitlesClip("output_subtitles.srt", make_textclip=subtitle_generator)

final = CompositeVideoClip(
    [video, subtitles.with_position(("center", video.h - 160))]
)

final.write_videofile(
    str(output_folder / "final_video_complete.mp4"),
    fps=fps,
    codec="libx264",
    audio_codec="aac",
    preset="ultrafast",  # HUGE speed improvement
    threads=4
)

MoviePy - Building video C:\Uni\GP\Code\Generation\Output_Videos\final_video_complete.mp4.
MoviePy - Writing audio in final_video_completeTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video C:\Uni\GP\Code\Generation\Output_Videos\final_video_complete.mp4



MoviePy - Done !
MoviePy - video ready C:\Uni\GP\Code\Generation\Output_Videos\final_video_complete.mp4
